[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataguirre/Curso-IA-Aplicada/blob/main/Semana%2011_Arquitectura_Transformers/transformers.ipynb)

# Fine tuning y Prompt Engineer

In [1]:
!pip install -U --quiet "transformers>=4.44.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.1" "peft>=0.12.0" sentencepiece
!pip install -U --quiet --no-cache-dir bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 13.0 MB/s eta 0:00:00


In [ ]:
import os, sys, time
print("Reiniciando el runtime para activar bitsandbytes…")
time.sleep(1)
os.kill(os.getpid(), 9)

Reiniciando el runtime para activar bitsandbytes…


In [1]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import classification_report
from datasets import Dataset, DatasetDict, ClassLabel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import json
import pandas as pd

# Fine-Tuning

## ¿Qué es el Fine-Tuning?
El fine-tuning (ajuste fino) es el proceso de entrenar un modelo preentrenado en una tarea específica, utilizando un conjunto de datos más pequeño y especializado.
En lugar de entrenar un modelo desde cero —lo cual requiere enormes cantidades de datos y cómputo— aprovechamos el conocimiento general adquirido durante el preentrenamiento y lo adaptamos a una tarea concreta.

### Fine-Tuning en Encoders

En los modelos encoder (como BERT, RoBERTa o BETO), el fine-tuning consiste en:

	1.	Tomar un encoder ya preentrenado sobre grandes corpus de texto (por ejemplo, Wikipedia o Common Crawl).
	2.	Añadir una capa de clasificación (o de otra tarea supervisada, como NER o QA).
	3.	Ajustar todos los pesos —o solo las últimas capas— con un dataset específico.

De esta forma, el modelo mantiene su conocimiento lingüístico general, pero aprende a reconocer patrones propios del dominio o tarea (por ejemplo, texto científico, económico o jurídico).

### ¿Para qué sirve?

El fine-tuning permite:

	1.	Mejorar el rendimiento del modelo en tareas especializadas (clasificación, NER, análisis de sentimientos, etc.).
	2.	Adaptar un modelo general a un dominio específico (por ejemplo, texto científico o técnico).
	3.	Reducir el costo computacional frente a entrenar un modelo desde cero.

En general, el fine-tuning convierte un modelo “genérico” en uno optimizado para un contexto o tarea concreta.


### ¿Que vamos a hacer?
En este bloque realizaremos el *fine-tuning* del modelo [`Flaglab/Sci-BETO-base`](https://huggingface.co/Flaglab/Sci-BETO-base)
utilizando el dataset [`PlanTL-GOB-ES/WikiCAT_esv2`](https://huggingface.co/datasets/PlanTL-GOB-ES/WikiCAT_esv2).

El objetivo será adaptar **Sci-BETO**, un *encoder* científico en español,
a la tarea de **clasificación temática** de artículos de Wikipedia, y luego evaluar su rendimiento mediante la métrica **F1-score**.

In [2]:
# Descargar contenido de google drive y abrirlo
import requests

train_url = "https://drive.google.com/uc?export=download&id=1qxHQ5kxlz1cIfnNGyW_DSlywmdN97mPj"
test_url = "https://drive.google.com/uc?export=download&id=15OsbqCrtnkc1uYGh_HxlMtbwn7BXbei-"

response = requests.get(train_url)
response.encoding = 'utf-8'
train_json = json.loads(response.text)

response = requests.get(test_url)
response.encoding = 'utf-8'
test_json = json.loads(response.text)

train = train_json["data"]
test = test_json["data"]

train = pd.DataFrame(train)
test = pd.DataFrame(test)


In [3]:
train

,sentence,label
0,La administración es una de las actividades hu...,Economía
1,El aprendizaje por la práctica o aprendizaje p...,Economía
2,La clase social es una forma de estratificació...,Economía
3,Las ayudas y subvenciones a las empresas const...,Economía
4,"Un documento protestado, o bien un ""protesto"" ...",Economía
...,...,...
1680,Silicon Alley[1]​ (derivada de 'Silicon Valley...,Ciencia_y_Tecnología
1681,En el lenguaje de los autores vinculados con l...,Ciencia_y_Tecnología
1682,La Organización para las Mujeres en Ciencia pa...,Ciencia_y_Tecnología
1683,"El término investigación y desarrollo, abrevia...",Ciencia_y_Tecnología


In [4]:
test

,sentence,label
0,"En estadística, un modelo probit es un tipo de...",Economía
1,El libro diario o libro de cuentas es un libro...,Economía
2,La tarifa diaria promedio (comúnmente conocida...,Economía
3,"En economía, el coste medio o costo medio es i...",Economía
4,"Para un individuo, el equivalente cierto C(p) ...",Economía
...,...,...
6711,"Una computadora de tubos de vacío, ahora denom...",Ciencia_y_Tecnología
6712,La ingeniería económica conlleva la valoración...,Ciencia_y_Tecnología
6713,La epistemología bayesiana es un enfoque forma...,Ciencia_y_Tecnología
6714,Se define como economía de bambú un sector eco...,Ciencia_y_Tecnología


In [5]:
# Labels a predecir
train['label'].unique()

array(['Economía', 'Entretenimiento', 'Historia', 'Humanidades',
       'Derecho', 'Matemáticas', 'Música', 'Filosofía', 'Política',
       'Religión', 'Deporte', 'Ciencia_y_Tecnología'], dtype=object)

In [6]:
train, eval = train_test_split(
    train,
    test_size=0.2,
    random_state=42,
    stratify=train["label"]
)

unique_labels = sorted(train["label"].unique())
label_encoder = ClassLabel(names=unique_labels)

def encode_labels(df):
    df = df.copy()
    df["label"] = df["label"].apply(label_encoder.str2int)
    return df

train_encoded_df = encode_labels(train)
eval_encoded_df = encode_labels(eval)
test_encoded_df = encode_labels(test)

# Convertir a Hugging Face Dataset
ds = DatasetDict({
    "train": Dataset.from_pandas(train_encoded_df),
    "validation": Dataset.from_pandas(eval_encoded_df),
    "test": Dataset.from_pandas(test_encoded_df),
})

### ¿Qué es un data collator?

Un data collator (ensamblador de datos) se encarga de convertir una lista de ejemplos individuales en un solo lote que puede procesar el modelo.
En NLP, los textos tienen longitudes variables, y los tensores deben tener la misma longitud dentro del lote.

Ahí entra el DataCollatorWithPadding:

	1.	Detecta la secuencia más larga del lote.
	2.	Rellena (pad) las demás con el token especial <pad> del tokenizer.
	3.	Devuelve tensores (input_ids, attention_mask, etc.) listos para enviar al modelo

In [7]:
import os, pandas as pd, numpy as np, torch
from datasets import Dataset
from transformers import (
    RobertaTokenizerFast, RobertaForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score

model = 'Flaglab/Sci-BETO-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model)

# Tokenizacion del dataset
def tokenize(example):
    return tokenizer(example["sentence"], truncation=True, max_length=512)

tokenized_ds = ds.map(tokenize, batched=True)

# Eliminar columnas innecesarias
for split in tokenized_ds:
    existing_cols = set(tokenized_ds[split].column_names)
    required_cols = {"input_ids", "token_type_ids", "attention_mask", "label"}
    to_remove = list(existing_cols - required_cols)
    if to_remove:
        tokenized_ds[split] = tokenized_ds[split].remove_columns(to_remove)

tokenized_ds.set_format("torch")

# Padding dinámico
data_collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/149 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

Map:   0%|          | 0/1348 [00:00<?, ? examples/s]

Map:   0%|          | 0/337 [00:00<?, ? examples/s]

Map:   0%|          | 0/6716 [00:00<?, ? examples/s]

In [ ]:
num_labels = len(label_encoder.names)
model = RobertaForSequenceClassification.from_pretrained(
    model,
    num_labels=num_labels
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")
    f1_weighted = f1_score(labels, preds, average="weighted")
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted,
    }

training_args = TrainingArguments(
    output_dir="./sci-beto-wikicat-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=0.00003,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.1,
    warmup_ratio=0.06,
    lr_scheduler_type='linear',
    logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at Flaglab/Sci-BETO-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-869631963.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,2.251100,0.974632,0.727003,0.672044,0.717054
2,0.760900,0.806785,0.753709,0.707015,0.747780


In [ ]:
from sklearn.metrics import classification_report

preds = trainer.predict(tokenized_ds["validation"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)

print(classification_report(y_true, y_pred, digits=4))

In [ ]:
preds = trainer.predict(tokenized_ds["test"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=-1)

print(classification_report(y_true, y_pred, digits=4))

In [26]:
import torch, gc

# Borrar el modelo y liberar memoria
del model
del trainer

# Limpia variables del sistema y GPU
gc.collect()
torch.cuda.empty_cache()

## LLMs

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Cuantizar el modelo
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Cargar tokenizer y modelo
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_cfg,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

### Prompting

Prompting es la forma en que los seres humanos pueden comunicarse con las inteligencias artificiales. Es una manera de decirle a la IA qué queremos y cómo lo queremos, generalmente utilizando palabras. La ingeniería de prompts es la tarea de encontrar la indicación (texto de entrada) que obtiene los mejores resultados de la IA.


### Estrategias básicas para Prompting
- Escribir instrucciones claras
- Use delimitadores para indicar claramente las distintas partes de la entrada. Los delimitadores pueden ser: ```, """, < >, `<tag> </tag>`, `:`


In [25]:
# Construir el prompt para mistral
def chat_prompt(system, user):
    """
    Construye el prompt con formato Instruct estilo Mistral.
    """
    return f"<s>[INST] <<SYS>>\n{system}\n<</SYS>>\n{user} [/INST]"


system = "Eres un asistente útil y conciso."
text = f"""
El aprendizaje automático (AA) o aprendizaje automatizado o aprendizaje \
de máquinas o aprendizaje computacional (del inglés, machine learning) \
es el subcampo de las ciencias de la computación y una rama de la inteligencia artificial, \
cuyo objetivo es desarrollar técnicas que permitan que las computadoras aprendan. \
Se dice que un agente aprende cuando su desempeño mejora con la experiencia y mediante el uso de datos; \
es decir, cuando la habilidad no estaba presente en su genotipo o rasgos de nacimiento. \
En el aprendizaje de máquinas un computador observa datos,  \
construye un modelo basado en esos datos y utiliza ese modelo a la vez como \
una hipótesis acerca del mundo y una pieza de software que puede resolver problemas. \
"""
user = f"""
Resume el texto delimitado por triple backticks en una sola oración y responde en español.
```{text}```
"""
prompt = chat_prompt(system, user)

# Tokenizar entrada
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generar salida
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decodificar respuesta
decoded_zero = tokenizer.decode(outputs[0], skip_special_tokens=True)

if '[/INST]' in decoded_zero:
    respuesta = decoded_zero.split('[/INST]')[-1].strip()
else:
    respuesta = decoded_zero.strip()

# Mostrar bien estructurado
print("=" * 80)
print("Prompt:\n")
print(prompt)
print("\n" + "=" * 80)
print("Respuesta:\n")
print(respuesta)
print("=" * 80)

Prompt:

<s>[INST] <<SYS>>
Eres un asistente útil y conciso.
<</SYS>>

Resume el texto delimitado por triple backticks en una sola oración y responde en español.
```
El aprendizaje automático (AA) o aprendizaje automatizado o aprendizaje de máquinas o aprendizaje computacional (del inglés, machine learning) es el subcampo de las ciencias de la computación y una rama de la inteligencia artificial, cuyo objetivo es desarrollar técnicas que permitan que las computadoras aprendan. Se dice que un agente aprende cuando su desempeño mejora con la experiencia y mediante el uso de datos; es decir, cuando la habilidad no estaba presente en su genotipo o rasgos de nacimiento. En el aprendizaje de máquinas un computador observa datos,  construye un modelo basado en esos datos y utiliza ese modelo a la vez como una hipótesis acerca del mundo y una pieza de software que puede resolver problemas. ```
 [/INST]

Respuesta:

El aprendizaje automático, también conocido como aprendizaje automatizado, apre

In [26]:
def llm_answer(prompt, tokenizer=tokenizer, model=model):
  inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

  with torch.no_grad():
      outputs = model.generate(
          **inputs,
          max_new_tokens=1024,
          do_sample=True,
          temperature=0.7,
          top_p=0.9,
          eos_token_id=tokenizer.eos_token_id,
          pad_token_id=tokenizer.eos_token_id,
      )

  decoded_zero = tokenizer.decode(outputs[0], skip_special_tokens=True)

  if '[/INST]' in decoded_zero:
      respuesta = decoded_zero.split('[/INST]')[-1].strip()
  else:
      respuesta = decoded_zero.strip()

  print("=" * 80)
  print("Prompt:\n")
  print(prompt)
  print("\n" + "=" * 80)
  print("Respuesta:\n")
  print(respuesta)
  print("=" * 80)
  return None

## Zero shot learning

Este modelo de lenguage es tan poderoso que para algunas tareas no es necesario especificarle ejemplos, a esto se le llama zero shot y es de hecho la razón por la cual se cree que estos LLMs tienen un entendimiento parcial del mundo, Veamos unos ejemplos.

In [27]:
system = "Eres un asistente útil y conciso."
user = "Traduce al inglés: 'La economía colombiana creció 3,2% en 2024.'"
prompt = chat_prompt(system, user)
llm_answer(prompt)

Prompt:

<s>[INST] <<SYS>>
Eres un asistente útil y conciso.
<</SYS>>
Traduce al inglés: 'La economía colombiana creció 3,2% en 2024.' [/INST]

Respuesta:

The Colombian economy grew 3.2% in 2024.


### Analisis de sentimientos

In [28]:
review = """ Increible!,
Compré esto para la observación de aves y es un binocular muy bueno.
No pesa, es fácil de enfocar y es útil para caminar por el parque o por el bosque.
También proporciona un archivo adjunto para el teléfono, por lo que la fotografía de aves se ha convertido en mi nuevo pasatiempo.
"""

user = f"""
¿Cuál es el sentimiento de la siguiente revisión del producto,
que se delimita con triple backticks?

Review text: '''{review}'''
"""

prompt = chat_prompt(system, user)
llm_answer(prompt)

Prompt:

<s>[INST] <<SYS>>
Eres un asistente útil y conciso.
<</SYS>>

¿Cuál es el sentimiento de la siguiente revisión del producto,
que se delimita con triple backticks?

Review text: ''' Increible!,
Compré esto para la observación de aves y es un binocular muy bueno.
No pesa, es fácil de enfocar y es útil para caminar por el parque o por el bosque.
También proporciona un archivo adjunto para el teléfono, por lo que la fotografía de aves se ha convertido en mi nuevo pasatiempo.
'''
 [/INST]

Respuesta:

La revisión describe un buenexperiencia con el producto. El revisor elogia su facilidad de uso, su buen rendimiento para observación de aves, y el valor añadido de poder tomar fotos con el archivo adjunto para el teléfono. En conjunto, el sentimiento de la revisión es positivo.


In [30]:
review = """ Maravilloso!
Tienen que comprarlo!!! Solo 90k. Me toco empeñar la casa para comprarlo, y no pagar la pension de mis hijos.
Vale la pena.
Increíble experiencia.
"""

user = f"""
¿Cuál es el sentimiento de la siguiente revisión del producto,
que se delimita con triple backticks?

Review text: '''{review}'''
"""

prompt = chat_prompt(system, user)
llm_answer(prompt)

Prompt:

<s>[INST] <<SYS>>
Eres un asistente útil y conciso.
<</SYS>>

¿Cuál es el sentimiento de la siguiente revisión del producto,
que se delimita con triple backticks?

Review text: ''' Maravilloso!
Tienen que comprarlo!!! Solo 90k. Me toco empeñar la casa para comprarlo, y no pagar la pension de mis hijos.
Vale la pena.
Increíble experiencia.
'''
 [/INST]

Respuesta:

The given product review expresses a positive sentiment towards the product. The reviewer is highly praising the product, describing it as "maravilloso" (wonderful or marvelous), and expressing a strong desire to purchase it despite the significant cost of 90k, which they had to borrow money to afford. They also emphasize the incredible experience they had with the product.


## Few Shot Learning

Vamos a construir un clasificador por medio de un modelo de lenguaje. Para eso vamos a darle un pequeño conjunto de ejemplos en el prompt. A esta estrategia se le conoce como "few shot".

In [31]:
user = f"""
[Texto]: Fred es un emprendedor en serie. Cofundador y director ejecutivo de Platform.sh, anteriormente cofundó Commerce Guys, un proveedor líder de comercio electrónico de Drupal. Su misión es garantizar que mientras continuamos en un viaje ambicioso para transformar profundamente la forma en que se usa y se percibe la computación en la nube, mantenemos los pies bien puestos en el suelo y continuamos con el rápido crecimiento que hemos disfrutado hasta ahora.
[Nombre]: Fred
[Puesto]: Co-fundador y CEO
[Empresa]: Platform.sh
###
[Texto]: Microsoft (la palabra es un acrónimo de "software de microcomputadora") fue fundado por Bill Gates el 4 de abril de 1975 para desarrollar y vender intérpretes BASIC para Altair 8800. Steve Ballmer reemplazó a Gates como director ejecutivo en 2000 y luego imaginó una estrategia de "dispositivos y servicios".
[Nombre]: Steve Ballmer
[Puesto]: director general
[Empresa]: Microsoft
###
[Texto]: Franck Riboud nació el 7 de noviembre de 1955 en Lyon. Es hijo de Antoine Riboud, el anterior director ejecutivo, que transformó al antiguo fabricante de vidrio europeo BSN Group en un actor líder en la industria alimentaria. Es el director general de Danone.
[Nombre]: Franck Riboud
[Puesto]: director general
[Empresa]: Danone
###
[Texto]: David Melvin es un profesional de servicios financieros y de inversión en CITIC CLSA con más de 30 años de experiencia en banca de inversión y capital privado. Actualmente es Consejero Senior de CITIC CLSA
"""

prompt = chat_prompt(system, user)
llm_answer(prompt)

Prompt:

<s>[INST] <<SYS>>
Eres un asistente útil y conciso.
<</SYS>>

[Texto]: Fred es un emprendedor en serie. Cofundador y director ejecutivo de Platform.sh, anteriormente cofundó Commerce Guys, un proveedor líder de comercio electrónico de Drupal. Su misión es garantizar que mientras continuamos en un viaje ambicioso para transformar profundamente la forma en que se usa y se percibe la computación en la nube, mantenemos los pies bien puestos en el suelo y continuamos con el rápido crecimiento que hemos disfrutado hasta ahora.
[Nombre]: Fred
[Puesto]: Co-fundador y CEO
[Empresa]: Platform.sh
###
[Texto]: Microsoft (la palabra es un acrónimo de "software de microcomputadora") fue fundado por Bill Gates el 4 de abril de 1975 para desarrollar y vender intérpretes BASIC para Altair 8800. Steve Ballmer reemplazó a Gates como director ejecutivo en 2000 y luego imaginó una estrategia de "dispositivos y servicios".
[Nombre]: Steve Ballmer
[Puesto]: director general
[Empresa]: Microsoft
###


# Bono

Utiliza el dataset WikiCAT para realizar el fine-tuning de otro modelo encoder en español o multilingüe, y compara su desempeño con Sci-BETO.

Adicionalmente, genera una evaluación few-shot o zero-shot usando un modelo LLM (por ejemplo, Mistral, visto en clase) para resolver el test set de WikiCAT.

Finalmente, compara las métricas de desempeño (F1-score) entre:

*   Sci-BETO
*   El otro encoder fine-tuneado
*   El LLM usado en few-shot/zero-shot